# Pipelines — Broadcasting
## Advanced Tutorial Problems with Complete Solutions (Set B)

In this notebook we will continue working with **generator-based pipelines**, **coroutines**, **filters**, **broadcasting**, and **resource-safe cleanup**.

The goal is not merely to present finished functions. Instead, every problem is developed in small, logical steps:

1. understand the requirement,
2. sketch the data flow,
3. implement one component at a time,
4. test each component in isolation,
5. connect the components,
6. verify the final behavior with assertions.

The examples use only the Python standard library, so the notebook is self-contained.

### What makes these problems advanced?

The basic broadcaster sends every incoming item to several targets. Real pipelines usually need more:

- malformed records must be separated from valid records;
- one record may match several routing rules;
- stateful consumers may maintain counters or rolling summaries;
- partial batches must be flushed when a pipeline closes;
- one failing target should not necessarily stop every other target;
- files must always be closed, even when an exception occurs;
- duplicate records may need a separate audit path.

We will solve each of these problems without loading the full dataset into memory.

### Notebook map

We will first build a small reusable coroutine toolkit. Then we will solve seven guided problems and one capstone problem:

1. Normalize data before broadcasting it to independent branches.
2. Send valid and invalid records to different destinations.
3. Route records using overlapping, configurable rules.
4. Broadcast records to stateful analytics consumers.
5. Batch records and correctly flush the final partial batch.
6. Isolate failures inside a broadcaster.
7. Build a context-managed pipeline that writes several CSV files.
8. Combine validation, normalization, deduplication, routing, metrics, and auditing.

### How to use this notebook

Run the cells from top to bottom.

Many sections deliberately pause before the solution and ask a design question. Try to predict the behavior before running the next code cell. The assertions are part of the solutions: they turn an informal example into an executable specification.

In [1]:
from __future__ import annotations

import csv
import shutil
from collections import Counter
from contextlib import contextmanager
from dataclasses import asdict, dataclass
from pathlib import Path
from pprint import pprint
from typing import Any, Callable, Iterable, Iterator, Mapping, MutableSequence, Sequence

## 1. Preparing a self-contained dataset

The original broadcasting example works with car data. We will keep that scenario, but this notebook creates its own files.

We will create:

- a **clean** CSV file for the first exercises;
- a **messy** CSV file containing invalid rows and a duplicate VIN for validation and auditing exercises.

Keeping generated files in one workspace makes cleanup easy and prevents accidental changes elsewhere.

In [2]:
WORKDIR = Path("pipeline_broadcasting_tutorial")

if WORKDIR.exists():
    shutil.rmtree(WORKDIR)
WORKDIR.mkdir()

HEADERS = ["car_make", "car_model", "model_year", "vin", "color"]

CLEAN_ROWS = [
    ["Ford", "Mustang", "1967", "1FATP8UH0H5100001", "Red"],
    ["Mercedes-Benz", "S-Class", "2012", "WDDUG8FB0CA100002", "Black"],
    ["Toyota", "RAV4", "2008", "JTMBFREV0D0100003", "Green"],
    ["Tesla", "Roadster", "2012", "5YJRE1A10A1100004", "Blue"],
    ["BMW", "M3", "2004", "WBSBL93454P100005", "Silver"],
    ["Ford", "F150", "1999", "1FTRX17L1XK100006", "Blue"],
    ["Audi", "A8", "2011", "WAUZZZ4H1BN100007", "Green"],
    ["Honda", "Civic", "1988", "JHMED635XJS100008", "Red"],
    ["Mercedes-Benz", "SLK", "2008", "WDBWK56F18F100009", "Blue"],
    ["Chevrolet", "Corvette", "1962", "20867S10000100010", "Teal"],
    ["Toyota", "Camry", "2010", "4T1BF3EK7AU100011", "Green"],
    ["Porsche", "911", "2001", "WP0AA29911S100012", "Black"],
    ["Ford", "Explorer", "2005", "1FMEU73E75U100013", "Blue"],
    ["Nissan", "Murano", "2012", "JN8AZ1MW4CW100014", "Pink"],
    ["BMW", "X5", "2009", "5UXFE435X9L100015", "Green"],
    ["Volkswagen", "Golf", "1986", "WVWZZZ19ZGW100016", "Blue"],
    ["Lexus", "RX", "2008", "2T2HK31U08C100017", "Green"],
    ["Ferrari", "F430", "2008", "ZFFEW58A780100018", "Red"],
    ["Ford", "Thunderbird", "1965", "5Y83Z100000100019", "Blue"],
    ["Honda", "Fit", "2011", "JHMGE8H55BC100020", "Green"],
    ["Audi", "A4", "2000", "WAUZZZ8D1YA100021", "Silver"],
    ["Toyota", "Corolla", "1995", "1NXAE09B4SZ100022", "White"],
    ["Mercedes-Benz", "C-Class", "2008", "WDDGF54X08F100023", "Green"],
    ["Subaru", "Legacy", "2011", "4S3BMBC65B3100024", "Blue"],
]

MESSY_ROWS = CLEAN_ROWS + [
    ["", "Unknown", "2001", "1BADMAKE000100025", "Red"],
    ["Ford", "Focus", "twenty-ten", "1FAHP3FN0AW100026", "Blue"],
    ["Toyota", "Prius", "2010", "", "Green"],
    ["Honda", "Accord", "1890", "1HGCP2F30AA100028", "Silver"],
    ["Audi", "Q5", "2027", "WA1LFAFP7BA100029", "Black"],
    ["BMW", "Z4", "2009", "NOT-A-VALID-VIN!!", ""],
    CLEAN_ROWS[6],  # duplicate VIN: Audi A8
]


def write_csv(path: Path, rows: Sequence[Sequence[str]]) -> None:
    with path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(HEADERS)
        writer.writerows(rows)


CLEAN_FILE = WORKDIR / "clean_car_data.csv"
MESSY_FILE = WORKDIR / "messy_car_data.csv"
write_csv(CLEAN_FILE, CLEAN_ROWS)
write_csv(MESSY_FILE, MESSY_ROWS)

print(CLEAN_FILE)
print(MESSY_FILE)

pipeline_broadcasting_tutorial\clean_car_data.csv
pipeline_broadcasting_tutorial\messy_car_data.csv


Let's inspect a few rows before defining any pipeline components.

At this stage the CSV reader returns strings. Converting and validating those strings will be a separate responsibility later.

In [3]:
with CLEAN_FILE.open(encoding="utf-8") as file:
    reader = csv.reader(file)
    for _ in range(5):
        print(next(reader))

['car_make', 'car_model', 'model_year', 'vin', 'color']
['Ford', 'Mustang', '1967', '1FATP8UH0H5100001', 'Red']
['Mercedes-Benz', 'S-Class', '2012', 'WDDUG8FB0CA100002', 'Black']
['Toyota', 'RAV4', '2008', 'JTMBFREV0D0100003', 'Green']
['Tesla', 'Roadster', '2012', '5YJRE1A10A1100004', 'Blue']


## 2. A typed record and lazy readers

Passing raw lists through a large pipeline makes code hard to read because expressions such as `row[2]` do not explain what the value means.

We will convert valid rows to a small immutable `Car` object. Immutability is useful in a broadcast pipeline: every branch receives the same object, and one branch cannot accidentally modify it for the other branches.

In [4]:
@dataclass(frozen=True, slots=True)
class Car:
    make: str
    model: str
    year: int
    vin: str
    color: str


def read_raw_rows(path: Path) -> Iterator[dict[str, str]]:
    """Yield CSV rows lazily as dictionaries."""
    with path.open(newline="", encoding="utf-8") as file:
        yield from csv.DictReader(file)


def read_clean_cars(path: Path = CLEAN_FILE) -> Iterator[Car]:
    """Parse the known-clean tutorial file lazily."""
    for row in read_raw_rows(path):
        yield Car(
            make=row["car_make"].strip(),
            model=row["car_model"].strip(),
            year=int(row["model_year"]),
            vin=row["vin"].strip(),
            color=row["color"].strip(),
        )

The reader is a generator. It does not build a list of every car.

We can prove that it is lazy by taking only three records.

In [5]:
cars = read_clean_cars()
for _ in range(3):
    print(next(cars))

Car(make='Ford', model='Mustang', year=1967, vin='1FATP8UH0H5100001', color='Red')
Car(make='Mercedes-Benz', model='S-Class', year=2012, vin='WDDUG8FB0CA100002', color='Black')
Car(make='Toyota', model='RAV4', year=2008, vin='JTMBFREV0D0100003', color='Green')


## 3. A small coroutine toolkit

A coroutine used as a pipeline target pauses at `yield` and waits for the next item sent with `.send(...)`.

A newly created generator must normally be primed with `next(generator)` before receiving a non-`None` value. We will hide that repetitive step in a decorator.

In [6]:
def coroutine(function: Callable[..., Iterator[Any]]) -> Callable[..., Iterator[Any]]:
    """Create and automatically prime a coroutine."""
    def start(*args: Any, **kwargs: Any) -> Iterator[Any]:
        generator = function(*args, **kwargs)
        next(generator)
        return generator
    return start

### A collector sink

For tutorial tests, writing every result to disk would be inconvenient. A collector sink appends every received item to a list supplied by the caller.

Because the caller owns the list, we can inspect it after the pipeline closes.

In [7]:
@coroutine
def collect_into(storage: MutableSequence[Any]) -> Iterator[None]:
    while True:
        item = yield
        storage.append(item)

### A filter stage

A filter receives every item but forwards only items for which its predicate returns `True`.

Notice the `finally` block. When the upstream stage closes this filter, the filter closes its target. This is **close propagation**: cleanup moves from the root of the pipeline to its endpoints.

In [8]:
@coroutine
def filter_coro(
    predicate: Callable[[Any], bool],
    target: Iterator[Any],
) -> Iterator[None]:
    try:
        while True:
            item = yield
            if predicate(item):
                target.send(item)
    finally:
        target.close()

### A transformation stage

Filtering decides *whether* an item continues. Mapping decides *what* continues.

Keeping these operations separate makes a pipeline easier to test and reuse.

In [9]:
@coroutine
def map_coro(
    transform: Callable[[Any], Any],
    target: Iterator[Any],
) -> Iterator[None]:
    try:
        while True:
            item = yield
            target.send(transform(item))
    finally:
        target.close()

### A broadcaster

The broadcaster sends the same item to every target.

This creates **fan-out**. A single input stream can feed file writers, counters, filters, or other broadcasters.

In [10]:
def close_unique(targets: Iterable[Iterator[Any]]) -> None:
    """Close each distinct target at most once."""
    closed_ids: set[int] = set()
    for target in targets:
        if id(target) not in closed_ids:
            closed_ids.add(id(target))
            target.close()


@coroutine
def broadcast(*targets: Iterator[Any]) -> Iterator[None]:
    try:
        while True:
            item = yield
            for target in targets:
                target.send(item)
    finally:
        close_unique(targets)

### A first smoke test

Before solving larger problems, let's verify the small components.

The pipeline below sends cars to two independent filters:

```text
                     +--> blue filter --> blue collector
source --> broadcast |
                     +--> Ford filter --> Ford collector
```

A blue Ford should appear in both outputs. Broadcasting does not mean exclusive routing.

In [11]:
blue_cars: list[Car] = []
ford_cars: list[Car] = []

pipe = broadcast(
    filter_coro(lambda car: car.color == "Blue", collect_into(blue_cars)),
    filter_coro(lambda car: car.make == "Ford", collect_into(ford_cars)),
)

for car in read_clean_cars():
    pipe.send(car)
pipe.close()

print("blue:", len(blue_cars))
print("Ford:", len(ford_cars))
print("blue Fords:", [car.model for car in blue_cars if car.make == "Ford"])

blue: 7
Ford: 4
blue Fords: ['F150', 'Explorer', 'Thunderbird']


In [12]:
assert all(car.color == "Blue" for car in blue_cars)
assert all(car.make == "Ford" for car in ford_cars)
assert {car.vin for car in blue_cars} & {car.vin for car in ford_cars}
print("Smoke test passed.")

Smoke test passed.


# Problem 1 — Normalize once, then broadcast

Suppose color names arrive with inconsistent capitalization and whitespace:

- `" green "`
- `"GREEN"`
- `"Green"`

We want every downstream branch to receive the same normalized representation.

A poor design would repeat normalization inside every predicate. A better design performs normalization **once before fan-out**.

### Step 1: Decide where normalization belongs

Compare these two shapes:

```text
source --> broadcast --> normalize --> filter
                    --> normalize --> filter
```

and

```text
source --> normalize --> broadcast --> filter
                                  --> filter
```

The second design avoids duplicated work and guarantees that every branch sees the same normalized object.

In [13]:
def normalize_car(car: Car) -> Car:
    return Car(
        make=" ".join(car.make.split()),
        model=" ".join(car.model.split()),
        year=car.year,
        vin=car.vin.strip().upper(),
        color=" ".join(car.color.split()).title(),
    )

messy_but_valid = Car(
    make="  Mercedes-Benz  ",
    model="  S-Class ",
    year=2012,
    vin=" wddtest0000000001 ",
    color=" GREEN ",
)

print(normalize_car(messy_but_valid))

Car(make='Mercedes-Benz', model='S-Class', year=2012, vin='WDDTEST0000000001', color='Green')


### Step 2: Build independent normalized branches

We will collect:

- green cars made in or after 2005;
- normalized Mercedes-Benz cars;
- every normalized car, for comparison.

In [14]:
normalized_all: list[Car] = []
modern_green: list[Car] = []
mercedes: list[Car] = []

normalized_fanout = broadcast(
    collect_into(normalized_all),
    filter_coro(
        lambda car: car.year >= 2005 and car.color == "Green",
        collect_into(modern_green),
    ),
    filter_coro(
        lambda car: car.make == "Mercedes-Benz",
        collect_into(mercedes),
    ),
)

pipe = map_coro(normalize_car, normalized_fanout)

### Step 3: Send data through the pipeline

For this exercise we deliberately alter the capitalization of a few cars before sending them. The pipeline should normalize them before any predicate runs.

In [15]:
for index, car in enumerate(read_clean_cars()):
    if index % 5 == 0:
        car = Car(car.make.lower(), car.model, car.year, car.vin.lower(), f" {car.color.upper()} ")
    pipe.send(car)
pipe.close()

print("all normalized:", len(normalized_all))
print("modern green:", [(car.make, car.model) for car in modern_green])
print("Mercedes-Benz:", [(car.model, car.color) for car in mercedes])

all normalized: 24
modern green: [('Toyota', 'RAV4'), ('Audi', 'A8'), ('toyota', 'Camry'), ('BMW', 'X5'), ('Lexus', 'RX'), ('Honda', 'Fit'), ('Mercedes-Benz', 'C-Class')]
Mercedes-Benz: [('S-Class', 'Black'), ('SLK', 'Blue'), ('C-Class', 'Green')]


### Solution discussion

The important decision was not the implementation of `title()` or `strip()`. The important decision was the **position** of the transformation.

Because `map_coro(normalize_car, ...)` is upstream of the broadcaster:

- normalization occurs once per record;
- every branch receives the normalized `Car`;
- predicates remain short and readable;
- branch behavior cannot disagree because of capitalization differences.

In [16]:
assert len(normalized_all) == len(CLEAN_ROWS)
assert all(car.color == car.color.title() for car in normalized_all)
assert all(car.make == "Mercedes-Benz" for car in mercedes)
assert all(car.year >= 2005 and car.color == "Green" for car in modern_green)
print("Problem 1 solution verified.")

Problem 1 solution verified.


# Problem 2 — Validation with a side output

The messy CSV file contains records that should not enter the main business pipeline.

We need a validator with two outputs:

```text
                          +--> valid target
raw rows --> validator ---|
                          +--> invalid-record target
```

The invalid path is sometimes called a **side output**, **dead-letter stream**, or **quarantine stream**.

### Step 1: Define a useful error object

Saving only the message `"invalid row"` would make debugging difficult.

A useful validation issue should contain:

- the original raw row;
- every error found in that row, not just the first one.

In [17]:
@dataclass(frozen=True, slots=True)
class ValidationIssue:
    raw: Mapping[str, str]
    errors: tuple[str, ...]


def validate_row(row: Mapping[str, str]) -> tuple[Car | None, ValidationIssue | None]:
    errors: list[str] = []

    make = row.get("car_make", "").strip()
    model = row.get("car_model", "").strip()
    year_text = row.get("model_year", "").strip()
    vin = row.get("vin", "").strip().upper()
    color = row.get("color", "").strip()

    if not make:
        errors.append("make is required")
    if not model:
        errors.append("model is required")
    if not color:
        errors.append("color is required")

    try:
        year = int(year_text)
    except ValueError:
        year = 0
        errors.append("year must be an integer")
    else:
        if not 1950 <= year <= 2026:
            errors.append("year must be between 1950 and 2026")

    if len(vin) != 17 or not vin.isalnum():
        errors.append("VIN must contain exactly 17 letters or digits")

    if errors:
        return None, ValidationIssue(dict(row), tuple(errors))

    return Car(make, model, year, vin, color), None

Let's test the pure validation function before placing it inside a coroutine.

Pure functions are easier to test because they do not depend on generator state.

In [18]:
example_invalid = {
    "car_make": "",
    "car_model": "Example",
    "model_year": "future",
    "vin": "bad",
    "color": "",
}

car, issue = validate_row(example_invalid)
print(car)
pprint(issue)

None
ValidationIssue(raw={'car_make': '',
                     'car_model': 'Example',
                     'color': '',
                     'model_year': 'future',
                     'vin': 'bad'},
                errors=('make is required',
                        'color is required',
                        'year must be an integer',
                        'VIN must contain exactly 17 letters or digits'))


### Step 2: Convert the pure function into a pipeline stage

The stage sends a `Car` to the valid target or a `ValidationIssue` to the invalid target.

It closes both targets when it is closed.

In [19]:
@coroutine
def validate_coro(
    valid_target: Iterator[Any],
    invalid_target: Iterator[Any],
) -> Iterator[None]:
    try:
        while True:
            row = yield
            car, issue = validate_row(row)
            if issue is None:
                valid_target.send(car)
            else:
                invalid_target.send(issue)
    finally:
        close_unique((valid_target, invalid_target))

### Step 3: Wire and run the validator

For now, both destinations are collectors. Later, the valid target could be a large routing pipeline and the invalid target could be a CSV or JSON audit sink.

In [20]:
valid_cars: list[Car] = []
validation_issues: list[ValidationIssue] = []

pipe = validate_coro(
    valid_target=collect_into(valid_cars),
    invalid_target=collect_into(validation_issues),
)

for raw_row in read_raw_rows(MESSY_FILE):
    pipe.send(raw_row)
pipe.close()

print("valid records:", len(valid_cars))
print("invalid records:", len(validation_issues))

valid records: 25
invalid records: 6


### Step 4: Inspect the side output

A single row can have several errors. Reporting all of them reduces repeated correction cycles.

In [21]:
for number, issue in enumerate(validation_issues, start=1):
    print(f"Issue {number}")
    print(" raw:", dict(issue.raw))
    print(" errors:", issue.errors)
    print()

Issue 1
 raw: {'car_make': '', 'car_model': 'Unknown', 'model_year': '2001', 'vin': '1BADMAKE000100025', 'color': 'Red'}
 errors: ('make is required',)

Issue 2
 raw: {'car_make': 'Ford', 'car_model': 'Focus', 'model_year': 'twenty-ten', 'vin': '1FAHP3FN0AW100026', 'color': 'Blue'}
 errors: ('year must be an integer',)

Issue 3
 raw: {'car_make': 'Toyota', 'car_model': 'Prius', 'model_year': '2010', 'vin': '', 'color': 'Green'}
 errors: ('VIN must contain exactly 17 letters or digits',)

Issue 4
 raw: {'car_make': 'Honda', 'car_model': 'Accord', 'model_year': '1890', 'vin': '1HGCP2F30AA100028', 'color': 'Silver'}
 errors: ('year must be between 1950 and 2026',)

Issue 5
 raw: {'car_make': 'Audi', 'car_model': 'Q5', 'model_year': '2027', 'vin': 'WA1LFAFP7BA100029', 'color': 'Black'}
 errors: ('year must be between 1950 and 2026',)

Issue 6
 raw: {'car_make': 'BMW', 'car_model': 'Z4', 'model_year': '2009', 'vin': 'NOT-A-VALID-VIN!!', 'color': ''}
 errors: ('color is required', 'VIN must 

### Solution discussion

The validator separates **control flow** from **business routing**:

- invalid data never reaches the main pipeline;
- invalid rows are not silently discarded;
- the side output preserves enough context for correction;
- the pipeline remains streaming and processes one row at a time.

In [22]:
assert len(valid_cars) == len(CLEAN_ROWS) + 1  # duplicate row is still structurally valid
assert len(validation_issues) == 6
assert any(len(issue.errors) > 1 for issue in validation_issues)
assert all(isinstance(car.year, int) for car in valid_cars)
print("Problem 2 solution verified.")

Problem 2 solution verified.


# Problem 3 — Configurable routing with overlapping rules

Hard-coding one filter variable per business rule becomes repetitive.

We want a router configured with rule objects. Each rule has:

- a name;
- a predicate;
- a target.

By default, a car should be sent to **every matching rule**. Cars that match no rule should be sent to an unmatched target.

### Step 1: Model a route

A route object makes the configuration explicit and allows us to include the rule name in error messages or metrics later.

In [23]:
@dataclass(frozen=True)
class Route:
    name: str
    predicate: Callable[[Car], bool]
    target: Iterator[Any]

### Step 2: Define router semantics

There are two common routing modes:

- **overlapping mode**: send to every matching route;
- **first-match mode**: stop after the first match.

The order of rules matters only in first-match mode.

In [24]:
@coroutine
def route_coro(
    routes: Sequence[Route],
    unmatched_target: Iterator[Any] | None = None,
    *,
    first_match: bool = False,
) -> Iterator[None]:
    try:
        while True:
            item = yield
            matched = False

            for route in routes:
                if route.predicate(item):
                    route.target.send(item)
                    matched = True
                    if first_match:
                        break

            if not matched and unmatched_target is not None:
                unmatched_target.send(item)
    finally:
        targets = [route.target for route in routes]
        if unmatched_target is not None:
            targets.append(unmatched_target)
        close_unique(targets)

### Step 3: Configure overlapping routes

We will create three routes:

- `classic`: before 1990;
- `luxury`: selected luxury makes;
- `modern_green`: green and 2005 or newer.

A classic Ferrari can match both `classic` and `luxury`. A modern green Mercedes-Benz can match both `luxury` and `modern_green`.

In [25]:
classic: list[Car] = []
luxury: list[Car] = []
modern_green_route: list[Car] = []
unmatched: list[Car] = []

LUXURY_MAKES = {"Audi", "BMW", "Ferrari", "Lexus", "Mercedes-Benz", "Porsche", "Tesla"}

routes = [
    Route("classic", lambda car: car.year < 1990, collect_into(classic)),
    Route("luxury", lambda car: car.make in LUXURY_MAKES, collect_into(luxury)),
    Route(
        "modern_green",
        lambda car: car.year >= 2005 and car.color == "Green",
        collect_into(modern_green_route),
    ),
]

pipe = route_coro(routes, collect_into(unmatched))
for car in read_clean_cars():
    pipe.send(car)
pipe.close()

### Step 4: Inspect overlap

We compare VIN sets because a VIN identifies one car in this dataset.

In [26]:
classic_vins = {car.vin for car in classic}
luxury_vins = {car.vin for car in luxury}
green_vins = {car.vin for car in modern_green_route}

print("classic:", [(car.make, car.model) for car in classic])
print("luxury count:", len(luxury))
print("modern green:", [(car.make, car.model) for car in modern_green_route])
print("unmatched count:", len(unmatched))
print("luxury AND modern green:", luxury_vins & green_vins)

classic: [('Ford', 'Mustang'), ('Honda', 'Civic'), ('Chevrolet', 'Corvette'), ('Volkswagen', 'Golf'), ('Ford', 'Thunderbird')]
luxury count: 11
modern green: [('Toyota', 'RAV4'), ('Audi', 'A8'), ('Toyota', 'Camry'), ('BMW', 'X5'), ('Lexus', 'RX'), ('Honda', 'Fit'), ('Mercedes-Benz', 'C-Class')]
unmatched count: 5
luxury AND modern green: {'5UXFE435X9L100015', 'WAUZZZ4H1BN100007', '2T2HK31U08C100017', 'WDDGF54X08F100023'}


### Solution discussion

This router is different from a chain of `if/elif` statements.

In overlapping mode, the router continues checking after a match. That is essential when categories are not mutually exclusive.

The unmatched target also prevents silent data loss. Every input is either routed somewhere or made visible as unmatched.

In [27]:
all_routed_or_unmatched = classic_vins | luxury_vins | green_vins | {car.vin for car in unmatched}
all_input_vins = {car.vin for car in read_clean_cars()}

assert all_routed_or_unmatched == all_input_vins
assert luxury_vins & green_vins
assert all(car.year < 1990 for car in classic)
assert all(car.make in LUXURY_MAKES for car in luxury)
print("Problem 3 solution verified.")

Problem 3 solution verified.


# Problem 4 — Broadcast to stateful analytics consumers

Not every endpoint writes records. Some endpoints maintain state.

We want to count cars by make and by color while also producing periodic snapshots after every five records.

The source should still be read only once.

### Step 1: A stateful counting sink

A coroutine's local variables survive between calls to `.send(...)`. This makes coroutines natural streaming aggregators.

The factory below returns both:

- the coroutine target;
- the `Counter` that contains its current state.

In [28]:
def counting_sink(
    key: Callable[[Any], Any],
) -> tuple[Iterator[Any], Counter[Any]]:
    counts: Counter[Any] = Counter()

    @coroutine
    def sink() -> Iterator[None]:
        while True:
            item = yield
            counts[key(item)] += 1

    return sink(), counts

### Step 2: A snapshotting counter

This stage sends a copy of its state downstream every `N` records. It also sends one final snapshot during cleanup.

The copied dictionary is important. Sending the same mutable `Counter` object repeatedly would cause every saved snapshot to appear to change later.

In [29]:
def snapshot_counter(
    key: Callable[[Any], Any],
    every: int,
    target: Iterator[Any],
) -> tuple[Iterator[Any], Counter[Any]]:
    if every <= 0:
        raise ValueError("every must be positive")

    counts: Counter[Any] = Counter()
    processed = 0

    @coroutine
    def counter_coro() -> Iterator[None]:
        nonlocal processed
        try:
            while True:
                item = yield
                processed += 1
                counts[key(item)] += 1
                if processed % every == 0:
                    target.send({
                        "processed": processed,
                        "final": False,
                        "counts": dict(counts),
                    })
        finally:
            target.send({
                "processed": processed,
                "final": True,
                "counts": dict(counts),
            })
            target.close()

    return counter_coro(), counts

### Step 3: Build one-pass analytics

The broadcaster sends every car to:

- a make counter;
- a color counter;
- a snapshotting counter.

In [30]:
make_target, make_counts = counting_sink(lambda car: car.make)
color_target, color_counts = counting_sink(lambda car: car.color)
snapshots: list[dict[str, Any]] = []
snapshot_target, snapshot_state = snapshot_counter(
    key=lambda car: car.make,
    every=5,
    target=collect_into(snapshots),
)

pipe = broadcast(make_target, color_target, snapshot_target)
for car in read_clean_cars():
    pipe.send(car)
pipe.close()

### Step 4: Inspect final state and snapshots

There are 24 records, so we expect snapshots after 5, 10, 15, and 20 records, followed by a final snapshot at 24.

In [31]:
print("Top makes:")
pprint(make_counts.most_common(5))

print("\nColors:")
pprint(color_counts)

print("\nSnapshot positions:", [snapshot["processed"] for snapshot in snapshots])
print("Final snapshot:")
pprint(snapshots[-1])

Top makes:
[('Ford', 4), ('Mercedes-Benz', 3), ('Toyota', 3), ('BMW', 2), ('Audi', 2)]

Colors:
Counter({'Green': 7,
         'Blue': 7,
         'Red': 3,
         'Black': 2,
         'Silver': 2,
         'Teal': 1,
         'Pink': 1,
         'White': 1})

Snapshot positions: [5, 10, 15, 20, 24]
Final snapshot:
{'counts': {'Audi': 2,
            'BMW': 2,
            'Chevrolet': 1,
            'Ferrari': 1,
            'Ford': 4,
            'Honda': 2,
            'Lexus': 1,
            'Mercedes-Benz': 3,
            'Nissan': 1,
            'Porsche': 1,
            'Subaru': 1,
            'Tesla': 1,
            'Toyota': 3,
            'Volkswagen': 1},
 'final': True,
 'processed': 24}


### Solution discussion

Broadcasting lets transactional and analytical work happen together. A future version could send one branch to a file, another to monitoring, and another to an alerting system.

The source remains lazy and is traversed exactly once.

In [32]:
assert sum(make_counts.values()) == len(CLEAN_ROWS)
assert sum(color_counts.values()) == len(CLEAN_ROWS)
assert [snapshot["processed"] for snapshot in snapshots] == [5, 10, 15, 20, 24]
assert snapshots[-1]["final"] is True
assert snapshots[-1]["counts"] == dict(snapshot_state)
print("Problem 4 solution verified.")

Problem 4 solution verified.


# Problem 5 — Batching and the importance of closing

Sending one record at a time may be inefficient for a database or remote API.

We want a batching stage that:

- collects exactly `batch_size` items;
- sends each full batch downstream;
- sends the final partial batch when the pipeline closes.

### Step 1: Think about the final partial batch

With a batch size of 3 and five input records, the desired output is:

```python
[[item1, item2, item3], [item4, item5]]
```

The second batch can be emitted only when either:

- more data arrives and fills it, or
- the stage is told that the stream has ended.

For a coroutine pipeline, `.close()` provides that end-of-stream signal.

In [33]:
@coroutine
def batch_coro(
    batch_size: int,
    target: Iterator[Any],
) -> Iterator[None]:
    if batch_size <= 0:
        raise ValueError("batch_size must be positive")

    batch: list[Any] = []
    try:
        while True:
            item = yield
            batch.append(item)
            if len(batch) == batch_size:
                target.send(batch.copy())
                batch.clear()
    finally:
        if batch:
            target.send(batch.copy())
        target.close()

### Step 2: Observe the pipeline before and after close

We will send five cars into batches of three. Before closing, only the full batch should be visible.

In [34]:
batches: list[list[Car]] = []
pipe = batch_coro(3, collect_into(batches))

first_five = []
for index, car in enumerate(read_clean_cars()):
    if index == 5:
        break
    first_five.append(car)
    pipe.send(car)

print("before close:", [len(batch) for batch in batches])
pipe.close()
print("after close:", [len(batch) for batch in batches])

before close: [3]
after close: [3, 2]


The final two records were not lost. They were held in the coroutine's local `batch` list and flushed by the `finally` block.

This is why forgetting to close a pipeline can produce an empty or incomplete output even though no exception occurred.

In [35]:
assert [len(batch) for batch in batches] == [3, 2]
assert [car.vin for batch in batches for car in batch] == [car.vin for car in first_five]
print("Problem 5 solution verified.")

Problem 5 solution verified.


### An additional batching example: broadcast batches, not records

The location of the broadcaster changes the unit of work.

```text
records --> batch --> broadcast --> destination A
                              --> destination B
```

Both destinations receive lists. This can reduce the number of calls made by expensive sinks.

In [36]:
batch_a: list[list[Car]] = []
batch_b: list[list[Car]] = []

pipe = batch_coro(
    7,
    broadcast(collect_into(batch_a), collect_into(batch_b)),
)
for car in read_clean_cars():
    pipe.send(car)
pipe.close()

print([len(batch) for batch in batch_a])
assert batch_a == batch_b
assert [len(batch) for batch in batch_a] == [7, 7, 7, 3]

[7, 7, 7, 3]


# Problem 6 — Failure isolation in a broadcaster

The simple broadcaster is fail-fast. If one target raises an exception, the exception interrupts the send loop.

That behavior is sometimes correct. In other systems, one optional target should be allowed to fail while critical targets continue.

We will first observe the fail-fast behavior, then build a resilient alternative.

### Step 1: Create a deliberately unreliable sink

This sink raises an exception when it receives a car from a selected make.

In [37]:
def flaky_sink(
    fail_make: str,
    received: MutableSequence[Car],
) -> Iterator[Any]:
    @coroutine
    def sink() -> Iterator[None]:
        while True:
            car = yield
            if car.make == fail_make:
                raise RuntimeError(f"simulated failure for {fail_make}")
            received.append(car)

    return sink()

### Step 2: Demonstrate the simple broadcaster's behavior

The target after the failing sink will not receive the failing record because iteration stops at the exception.

We catch the exception so the notebook can continue.

In [38]:
first_target: list[Car] = []
flaky_received: list[Car] = []
last_target: list[Car] = []

pipe = broadcast(
    collect_into(first_target),
    flaky_sink("Tesla", flaky_received),
    collect_into(last_target),
)

try:
    for car in read_clean_cars():
        pipe.send(car)
except RuntimeError as exc:
    print("caught:", exc)

print("first target records:", len(first_target))
print("last target records:", len(last_target))
print("last car in first target:", first_target[-1].make)
print("last car in last target:", last_target[-1].make)

caught: simulated failure for Tesla
first target records: 4
last target records: 3
last car in first target: Tesla
last car in last target: Toyota


The first target received the Tesla before the failing sink raised. The last target did not receive that same Tesla.

This means target order can affect partial delivery in a fail-fast broadcaster.

### Step 3: Build a resilient broadcaster

The resilient version records the failure, removes the failed target, and continues sending the current item to the remaining targets.

This is an explicit policy choice. It should not silently replace fail-fast behavior everywhere.

In [39]:
@dataclass(frozen=True)
class DeliveryFailure:
    target_name: str
    item: Any
    error_type: str
    message: str


@coroutine
def resilient_broadcast(
    named_targets: Sequence[tuple[str, Iterator[Any]]],
    failures: MutableSequence[DeliveryFailure],
) -> Iterator[None]:
    active = list(named_targets)
    try:
        while True:
            item = yield
            survivors: list[tuple[str, Iterator[Any]]] = []

            for name, target in active:
                try:
                    target.send(item)
                except Exception as exc:
                    failures.append(DeliveryFailure(
                        target_name=name,
                        item=item,
                        error_type=type(exc).__name__,
                        message=str(exc),
                    ))
                    target.close()
                else:
                    survivors.append((name, target))

            active = survivors
    finally:
        close_unique(target for _, target in active)

### Step 4: Test failure isolation

The healthy targets should receive all 24 records. The failing target should receive records only until the first Tesla.

In [40]:
healthy_a: list[Car] = []
healthy_b: list[Car] = []
unreliable_records: list[Car] = []
failures: list[DeliveryFailure] = []

pipe = resilient_broadcast(
    [
        ("healthy-a", collect_into(healthy_a)),
        ("optional-tesla-sensitive", flaky_sink("Tesla", unreliable_records)),
        ("healthy-b", collect_into(healthy_b)),
    ],
    failures,
)

for car in read_clean_cars():
    pipe.send(car)
pipe.close()

print("healthy A:", len(healthy_a))
print("healthy B:", len(healthy_b))
print("unreliable before failure:", len(unreliable_records))
pprint(failures)

healthy A: 24
healthy B: 24
unreliable before failure: 3
[DeliveryFailure(target_name='optional-tesla-sensitive',
                 item=Car(make='Tesla',
                          model='Roadster',
                          year=2012,
                          vin='5YJRE1A10A1100004',
                          color='Blue'),
                 error_type='RuntimeError',
                 message='simulated failure for Tesla')]


### Solution discussion

A resilient broadcaster needs more than a `try/except`:

- failures must be observable;
- the failed target should be removed, otherwise every later item repeats the same failure;
- healthy targets should still receive the current item;
- remaining targets must still be closed.

For financial or transactional systems, fail-fast may still be safer. The correct policy depends on the meaning of each branch.

In [41]:
assert len(healthy_a) == len(CLEAN_ROWS)
assert len(healthy_b) == len(CLEAN_ROWS)
assert healthy_a == healthy_b
assert len(failures) == 1
assert failures[0].target_name == "optional-tesla-sensitive"
assert failures[0].item.make == "Tesla"
print("Problem 6 solution verified.")

Problem 6 solution verified.


# Problem 7 — Resource-safe CSV broadcasting

Now we will replace collector sinks with real CSV files.

Requirements:

- create separate files for blue cars, modern green cars, and luxury cars;
- allow overlap;
- write a header to each file;
- close every file automatically;
- expose one root coroutine to the consumer.

### Step 1: A CSV sink

The sink opens the file once and keeps it open while records arrive.

The `with` block closes the file when the coroutine is closed or when an exception unwinds the generator.

In [42]:
CAR_FIELDS = ["make", "model", "year", "vin", "color"]


@coroutine
def csv_car_sink(path: Path) -> Iterator[None]:
    with path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=CAR_FIELDS)
        writer.writeheader()

        while True:
            car = yield
            writer.writerow(asdict(car))

### Step 2: Put pipeline construction in a context manager

The caller should not need to remember which nested coroutine owns which file.

The context manager yields the root target and guarantees that `root.close()` runs.

In [43]:
@contextmanager
def file_routing_pipeline(output_dir: Path) -> Iterator[Iterator[Any]]:
    output_dir.mkdir(parents=True, exist_ok=True)

    blue_sink = csv_car_sink(output_dir / "blue.csv")
    green_sink = csv_car_sink(output_dir / "modern_green.csv")
    luxury_sink = csv_car_sink(output_dir / "luxury.csv")

    root = broadcast(
        filter_coro(lambda car: car.color == "Blue", blue_sink),
        filter_coro(lambda car: car.year >= 2005 and car.color == "Green", green_sink),
        filter_coro(lambda car: car.make in LUXURY_MAKES, luxury_sink),
    )

    try:
        yield root
    finally:
        root.close()

### Step 3: Run the pipeline

The consumer only sees one root coroutine. Closing is handled by the `with` statement.

In [44]:
ROUTED_DIR = WORKDIR / "problem_7_outputs"

with file_routing_pipeline(ROUTED_DIR) as pipe:
    for car in read_clean_cars():
        pipe.send(car)

print(sorted(path.name for path in ROUTED_DIR.iterdir()))

['blue.csv', 'luxury.csv', 'modern_green.csv']


### Step 4: Read the outputs back

Reading the files after the context manager exits also confirms that buffered data has been flushed.

In [45]:
def read_output(path: Path) -> list[dict[str, str]]:
    with path.open(newline="", encoding="utf-8") as file:
        return list(csv.DictReader(file))


blue_output = read_output(ROUTED_DIR / "blue.csv")
green_output = read_output(ROUTED_DIR / "modern_green.csv")
luxury_output = read_output(ROUTED_DIR / "luxury.csv")

print("blue rows:", len(blue_output))
print("modern green rows:", len(green_output))
print("luxury rows:", len(luxury_output))
print("example luxury row:")
pprint(luxury_output[0])

blue rows: 7
modern green rows: 7
luxury rows: 11
example luxury row:
{'color': 'Black',
 'make': 'Mercedes-Benz',
 'model': 'S-Class',
 'vin': 'WDDUG8FB0CA100002',
 'year': '2012'}


### Solution discussion

The context manager defines a clear ownership boundary:

- it constructs all sinks and intermediate stages;
- it yields only the root;
- it closes the root exactly once;
- close propagation flushes and closes all files.

This is the same design principle used by database sessions, file handles, locks, and network connections.

In [46]:
assert all(row["color"] == "Blue" for row in blue_output)
assert all(int(row["year"]) >= 2005 and row["color"] == "Green" for row in green_output)
assert all(row["make"] in LUXURY_MAKES for row in luxury_output)
assert any(row["make"] == "Mercedes-Benz" and row["color"] == "Green" for row in luxury_output)
print("Problem 7 solution verified.")

Problem 7 solution verified.


# Capstone Problem — An audit-ready streaming router

We will now combine the ideas from the notebook.

The input is the messy CSV file. The finished pipeline must:

1. validate raw rows;
2. save invalid rows to an in-memory audit list;
3. normalize valid cars;
4. detect duplicate VINs;
5. save duplicates to a separate audit list;
6. count unique cars by make;
7. route unique cars to overlapping CSV destinations;
8. save cars matching no route to an unmatched CSV file;
9. close every resource automatically.

### Capstone architecture

```text
raw CSV rows
     |
     v
 validation --------------------------> invalid audit list
     |
     v
 normalization
     |
     v
 deduplication -----------------------> duplicate audit list
     |
     v
 broadcast ---------------------------> make counter
     |
     v
 rule router --> classic.csv
             --> luxury.csv
             --> modern_green.csv
             --> unmatched.csv
```

The order matters. Deduplication happens after normalization and validation, so the key is a clean uppercase VIN.

### Step 1: Add a deduplication stage

We use a set of previously seen keys. The set grows with the number of unique VINs, which is acceptable for this tutorial.

For an unbounded production stream, a retention policy, database, or probabilistic structure might be required.

In [47]:
@coroutine
def deduplicate_coro(
    key: Callable[[Any], Any],
    unique_target: Iterator[Any],
    duplicate_target: Iterator[Any],
) -> Iterator[None]:
    seen: set[Any] = set()
    try:
        while True:
            item = yield
            item_key = key(item)
            if item_key in seen:
                duplicate_target.send(item)
            else:
                seen.add(item_key)
                unique_target.send(item)
    finally:
        close_unique((unique_target, duplicate_target))

### Step 2: Create a report object

The pipeline will write routed data to files, but audits and metrics should also be easy to inspect after the context manager exits.

In [48]:
@dataclass
class AuditReport:
    invalid_rows: list[ValidationIssue]
    duplicate_cars: list[Car]
    make_counts: Counter[str]
    output_dir: Path

### Step 3: Assemble the complete pipeline

Construction proceeds from the endpoints backward toward the root:

1. create file sinks;
2. create route objects;
3. create the router;
4. broadcast unique cars to the router and the counter;
5. add deduplication;
6. add normalization;
7. add validation.

Building backward is common in push-based coroutine pipelines because every stage needs to know its downstream target when it is created.

In [49]:
@contextmanager
def audit_pipeline(output_dir: Path) -> Iterator[tuple[Iterator[Any], AuditReport]]:
    output_dir.mkdir(parents=True, exist_ok=True)

    invalid_rows: list[ValidationIssue] = []
    duplicate_cars: list[Car] = []

    make_target, make_counts = counting_sink(lambda car: car.make)

    route_targets = {
        "classic": csv_car_sink(output_dir / "classic.csv"),
        "luxury": csv_car_sink(output_dir / "luxury.csv"),
        "modern_green": csv_car_sink(output_dir / "modern_green.csv"),
    }
    unmatched_target = csv_car_sink(output_dir / "unmatched.csv")

    router = route_coro(
        [
            Route("classic", lambda car: car.year < 1990, route_targets["classic"]),
            Route("luxury", lambda car: car.make in LUXURY_MAKES, route_targets["luxury"]),
            Route(
                "modern_green",
                lambda car: car.year >= 2005 and car.color == "Green",
                route_targets["modern_green"],
            ),
        ],
        unmatched_target,
    )

    unique_fanout = broadcast(router, make_target)
    deduplicator = deduplicate_coro(
        key=lambda car: car.vin,
        unique_target=unique_fanout,
        duplicate_target=collect_into(duplicate_cars),
    )
    normalizer = map_coro(normalize_car, deduplicator)
    root = validate_coro(
        valid_target=normalizer,
        invalid_target=collect_into(invalid_rows),
    )

    report = AuditReport(invalid_rows, duplicate_cars, make_counts, output_dir)

    try:
        yield root, report
    finally:
        root.close()

### Step 4: Run the capstone pipeline

The report object is created before processing. Its lists and counter are updated by the live coroutine graph.

In [50]:
CAPSTONE_DIR = WORKDIR / "capstone_outputs"

with audit_pipeline(CAPSTONE_DIR) as (pipe, report):
    for raw_row in read_raw_rows(MESSY_FILE):
        pipe.send(raw_row)

print("invalid rows:", len(report.invalid_rows))
print("duplicates:", len(report.duplicate_cars))
print("unique valid cars:", sum(report.make_counts.values()))
print("output files:", sorted(path.name for path in CAPSTONE_DIR.iterdir()))

invalid rows: 6
duplicates: 1
unique valid cars: 24
output files: ['classic.csv', 'luxury.csv', 'modern_green.csv', 'unmatched.csv']


### Step 5: Inspect audits and metrics

Invalid rows and duplicate rows are different problems:

- an invalid row could not become a trustworthy `Car`;
- a duplicate row is structurally valid but repeats a previously seen business key.

Keeping them separate allows different remediation processes.

In [51]:
print("First validation issue:")
pprint(report.invalid_rows[0])

print("\nDuplicate cars:")
pprint(report.duplicate_cars)

print("\nMake counts for unique valid cars:")
pprint(report.make_counts.most_common())

First validation issue:
ValidationIssue(raw={'car_make': '',
                     'car_model': 'Unknown',
                     'color': 'Red',
                     'model_year': '2001',
                     'vin': '1BADMAKE000100025'},
                errors=('make is required',))

Duplicate cars:
[Car(make='Audi',
     model='A8',
     year=2011,
     vin='WAUZZZ4H1BN100007',
     color='Green')]

Make counts for unique valid cars:
[('Ford', 4),
 ('Mercedes-Benz', 3),
 ('Toyota', 3),
 ('BMW', 2),
 ('Audi', 2),
 ('Honda', 2),
 ('Tesla', 1),
 ('Chevrolet', 1),
 ('Porsche', 1),
 ('Nissan', 1),
 ('Volkswagen', 1),
 ('Lexus', 1),
 ('Ferrari', 1),
 ('Subaru', 1)]


### Step 6: Inspect routed output files

The router uses overlapping semantics. Therefore, the sum of routed file lengths may be larger than the number of unique cars.

The unmatched file contains only cars that matched none of the three rules.

In [52]:
capstone_outputs = {
    path.stem: read_output(path)
    for path in sorted(CAPSTONE_DIR.glob("*.csv"))
}

for name, rows in capstone_outputs.items():
    print(f"{name:>12}: {len(rows)}")

luxury_green_vins = {
    row["vin"] for row in capstone_outputs["luxury"]
} & {
    row["vin"] for row in capstone_outputs["modern_green"]
}

print("\nVINs routed to both luxury and modern_green:")
pprint(luxury_green_vins)

     classic: 5
      luxury: 11
modern_green: 7
   unmatched: 5

VINs routed to both luxury and modern_green:
{'2T2HK31U08C100017',
 '5UXFE435X9L100015',
 'WAUZZZ4H1BN100007',
 'WDDGF54X08F100023'}


### Step 7: Final executable verification

These assertions describe the most important end-to-end guarantees:

- six malformed rows are quarantined;
- the repeated Audi is identified as one duplicate;
- duplicate data is not counted twice;
- routed files contain only records satisfying their own rules;
- unmatched records satisfy none of the routing predicates;
- at least one record appears in two overlapping destinations.

In [53]:
assert len(report.invalid_rows) == 6
assert len(report.duplicate_cars) == 1
assert report.duplicate_cars[0].make == "Audi"
assert sum(report.make_counts.values()) == len(CLEAN_ROWS)

assert all(int(row["year"]) < 1990 for row in capstone_outputs["classic"])
assert all(row["make"] in LUXURY_MAKES for row in capstone_outputs["luxury"])
assert all(
    int(row["year"]) >= 2005 and row["color"] == "Green"
    for row in capstone_outputs["modern_green"]
)

for row in capstone_outputs["unmatched"]:
    is_classic = int(row["year"]) < 1990
    is_luxury = row["make"] in LUXURY_MAKES
    is_modern_green = int(row["year"]) >= 2005 and row["color"] == "Green"
    assert not (is_classic or is_luxury or is_modern_green)

assert luxury_green_vins
print("Capstone solution verified.")

Capstone solution verified.


# Additional micro-problems

The following shorter exercises reinforce important edge cases.

### Micro-problem A: first-match routing

Re-run a router in first-match mode. Predict what happens to a green Mercedes-Benz when the `luxury` route appears before `modern_green`.

In [54]:
first_classic: list[Car] = []
first_luxury: list[Car] = []
first_green: list[Car] = []
first_unmatched: list[Car] = []

first_match_routes = [
    Route("classic", lambda car: car.year < 1990, collect_into(first_classic)),
    Route("luxury", lambda car: car.make in LUXURY_MAKES, collect_into(first_luxury)),
    Route("modern_green", lambda car: car.year >= 2005 and car.color == "Green", collect_into(first_green)),
]

pipe = route_coro(first_match_routes, collect_into(first_unmatched), first_match=True)
for car in read_clean_cars():
    pipe.send(car)
pipe.close()

print("first-match modern green:", [(car.make, car.model) for car in first_green])
assert not ({car.vin for car in first_luxury} & {car.vin for car in first_green})

first-match modern green: [('Toyota', 'RAV4'), ('Toyota', 'Camry'), ('Honda', 'Fit')]


In first-match mode, the categories become exclusive. A luxury green car stops at the luxury route and never reaches the later green route.

This demonstrates why routing semantics must be documented explicitly.

### Micro-problem B: transformation isolation

Because `Car` is immutable, a branch cannot mutate the shared record. A branch that needs a modified representation should create a new object with a mapping stage.

In [55]:
uppercase_models: list[Car] = []
original_models: list[Car] = []

uppercase_branch = map_coro(
    lambda car: Car(car.make, car.model.upper(), car.year, car.vin, car.color),
    collect_into(uppercase_models),
)

pipe = broadcast(uppercase_branch, collect_into(original_models))
for car in read_clean_cars():
    pipe.send(car)
pipe.close()

assert uppercase_models[0].model == original_models[0].model.upper()
assert original_models[0].model == "Mustang"
print(uppercase_models[0])
print(original_models[0])

Car(make='Ford', model='MUSTANG', year=1967, vin='1FATP8UH0H5100001', color='Red')
Car(make='Ford', model='Mustang', year=1967, vin='1FATP8UH0H5100001', color='Red')


### Micro-problem C: empty input

A well-designed pipeline should close cleanly even when it receives no records. File sinks should still contain headers, and finalizing stages should not fail.

In [56]:
EMPTY_DIR = WORKDIR / "empty_output"

with file_routing_pipeline(EMPTY_DIR):
    pass

for path in sorted(EMPTY_DIR.glob("*.csv")):
    rows = read_output(path)
    print(path.name, rows)
    assert rows == []

blue.csv []
luxury.csv []
modern_green.csv []


# Final recap

The central idea is simple: a broadcaster sends one item to several targets. The advanced behavior comes from how we compose stages around that broadcaster.

We used the following best practices:

- **single responsibility:** parsing, validation, normalization, filtering, batching, and writing are separate;
- **pure functions first:** validation and normalization can be tested without a live pipeline;
- **immutable records:** branches cannot accidentally interfere with one another;
- **explicit routing semantics:** overlapping and first-match modes are distinct;
- **observable failures:** resilient delivery records failures instead of hiding them;
- **close propagation:** final batches and buffered files are flushed;
- **context-managed ownership:** consumers cannot forget cleanup;
- **executable verification:** assertions document expected behavior.

These patterns generalize beyond CSV files. The same structure can feed databases, APIs, log processors, event streams, metrics, and audit systems.